# 03 - ETL dan validasi PostgreSQL 15 / Parquet
Jalankan notebook 01 dan 02 lebih dahulu, siapkan PostgreSQL, lalu isi variabel koneksi di .env. Kredensial tidak dicetak. Pemuatan tabel proyek bersifat atomik dan dapat diulang; kegagalan membatalkan transaksi.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd() if (Path.cwd()/"sales_project.py").exists() else Path.cwd().parent
assert (ROOT/"sales_project.py").exists(), "Buka notebook dari folder proyek/notebooks"
sys.path.insert(0,str(ROOT))
os.chdir(ROOT)
java_candidates = list((ROOT/".runtime/java").glob("*/bin/java.exe"))
if java_candidates:
    os.environ["JAVA_HOME"] = str(java_candidates[0].parent.parent)
os.environ["PYSPARK_PYTHON"] = sys.executable
from sales_project import *
print("Project:", ROOT.name)


Project: big-data-sales-analytics


In [2]:
db_result = postgres_stage()
display(db_result)

{
  "server_version": "PostgreSQL 15.19, compiled by Visual C++ build 1944, 64-bit",
  "rows": 100000,
  "units_sold": 551507,
  "net_sales": "54856691900.00",
  "formula_mismatch": 0,
  "exact_source_fields_match": true,
  "aggregate_matches": {
    "month": true,
    "branch": true,
    "category": true,
    "payment_method": true
  },
  "load_and_validation_seconds": 2.247693799989065,
  "audit_id": 2,
  "status": "SUCCESS"
}


{'server_version': 'PostgreSQL 15.19, compiled by Visual C++ build 1944, 64-bit',
 'rows': 100000,
 'units_sold': 551507,
 'net_sales': '54856691900.00',
 'formula_mismatch': 0,
 'exact_source_fields_match': True,
 'aggregate_matches': {'month': True,
  'branch': True,
  'category': True,
  'payment_method': True},
 'load_and_validation_seconds': 2.247693799989065,
 'audit_id': 2,
 'status': 'SUCCESS'}

In [3]:
import pandas as pd
display(pd.read_csv(OUT/"validation/postgres_audit.csv"))
display(pd.read_parquet(OUT/"parquet/sales_transactions").head())

,audit_id,job_name,started_at,finished_at,input_rows,output_rows,status
0,1,verified_sales_etl,2026-10-21 14:01:40.400045+07:00,2026-10-21 14:01:42.675738+07:00,100000,100000,SUCCESS
1,2,verified_sales_etl,2026-10-21 14:01:45.225446+07:00,2026-10-21 14:01:47.476162+07:00,100000,100000,SUCCESS


,transaction_id,transaction_date,branch,category,product_name,quantity,unit_price,discount_rate,gross_sales,net_sales,payment_method,customer_segment,calculated_gross_sales,calculated_net_sales,discount_amount,month,basket_band
0,1,2026-04-13,Denpasar,Kebutuhan Rumah,Botol Minum,2,104000.00,0.1500,208000.00,176800.00,Transfer,Retail,208000.00,176800.00,31200.00,2026-04,Low
1,2,2026-04-10,Surabaya,Olahraga,Raket,6,138000.00,0.0500,828000.00,786600.00,Kartu,Retail,828000.00,786600.00,41400.00,2026-04,Medium
2,3,2025-07-06,Bandung,Olahraga,Bola,2,92000.00,0.2000,184000.00,147200.00,E-Wallet,Retail,184000.00,147200.00,36800.00,2025-07,Low
3,4,2025-05-24,Semarang,Olahraga,Bola,7,158000.00,0.0500,1106000.00,1050700.00,Tunai,Retail,1106000.00,1050700.00,55300.00,2025-05,High
4,5,2026-05-09,Jakarta,Fashion,Jaket,4,170000.00,0.0000,680000.00,680000.00,E-Wallet,Corporate,680000.00,680000.00,0.00,2026-05,Medium


## Performa
Pengujian 10k, 100k, dan 1 juta baris dijalankan lewat `python sales_project.py benchmark`. Tiga ulangan, median; Spark dipaksa melakukan action agar lazy evaluation tidak menghasilkan waktu semu. Data 1 juta adalah pengulangan sumber dengan ID baru, bukan pengamatan independen.

In [4]:
if (OUT/"performance_median.csv").exists():
    display(pd.read_csv(OUT/"performance_median.csv"))
    print((OUT/"performance_method.json").read_text(encoding="utf-8"))
else:
    print("Jalankan benchmark untuk membuat tabel waktu aktual.")

,engine,rows,read_seconds,transform_aggregate_seconds,driver_collect_seconds,write_seconds,reread_seconds,total_seconds
0,pandas,10000,0.010142,0.023358,0.001454,0.010373,0.010319,0.055247
1,pandas,100000,0.078399,0.203015,0.021639,0.061606,0.029435,0.379806
2,pandas,1000000,0.632260,2.116913,0.227300,0.561815,0.216240,3.723863
3,spark,10000,0.162665,0.255739,0.091841,0.010887,0.010036,0.531316
4,spark,100000,0.259447,0.338920,0.603287,0.079853,0.029427,1.325640
5,spark,1000000,1.273069,1.103107,6.165351,0.748586,0.216098,9.552171


{
  "repeats": 3,
  "sizes": [
    10000,
    100000,
    1000000
  ],
  "spark_version": "3.5.3",
  "memory_note": "Spark driver 2 GiB; 2 local worker threads; shared Windows machine; filesystem cache not flushed.",
  "scaling": "10k first rows; 100k original; 1m repeated source rows with reassigned unique IDs. Scaling data is not new independent observations.",
  "protocol": "One warmup action excluded. Timed CSV read, date/net transformation plus month-branch-category aggregation, driver collection, Arrow Parquet write, Pandas reread. Spark cache/count/collect forces computation. Sequential runs; median of 3. Same storage writer for both engines. Small-data local lab, not cluster throughput.",
  "runtime": {
    "executed_at_utc": "2026-10-21T07:00:01.350418+00:00",
    "python": "3.12.14 (main, Aug 25 2026, 14:01:42) [MSC v.1944 64 bit (AMD64)]",
    "platform": "Windows-11-10.0.26300-SP0",
    "cpu": "Intel64 Family 6 Model 197 Stepping 2, GenuineIntel",
    "logical_cpus": 16,
  